In [1]:
# Please edit the path according to your PC environment
# When setting up a new environment, run this program after executing "data_downloader.ipynb".

from google.colab import drive
drive.mount('/content/drive')

!ls '/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/'
%cd "/content/drive/MyDrive/counterfactual_pitch_sequence_analysis/"

Mounted at /content/drive
data  dataset  models  program	results
/content/drive/MyDrive/counterfactual_pitch_sequence_analysis


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path("./data/pitch_sequence")
OUTPUT_DIR = Path("./data/slg_prob")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2018, 2019, 2022, 2023, 2024]

TARGET_COLUMN = "estimated_slg_using_speedangle"
SPEED_COLUMN = "effective_speed"

# Define & devide strike zone
N_X = 6; N_Z = 6
ZONE_X_MIN = -0.83
ZONE_X_MAX = 0.83
ZONE_Z_MIN = 0.0
ZONE_Z_MAX = 1.0

X_CENTER = (ZONE_X_MIN + ZONE_X_MAX) / 2
Z_CENTER = (ZONE_Z_MIN + ZONE_Z_MAX) / 2

# Define ball zone
BALL_X_MIN = ZONE_X_MIN - (ZONE_X_MAX - ZONE_X_MIN)
BALL_X_MAX = ZONE_X_MAX + (ZONE_X_MAX - ZONE_X_MIN)
BALL_Z_MIN = ZONE_Z_MIN - (ZONE_Z_MAX - ZONE_Z_MIN)
BALL_Z_MAX = ZONE_Z_MAX + (ZONE_Z_MAX - ZONE_Z_MIN)

In [5]:
# Define helper functions
def add_plate_z_norm(df):
    df = df.copy()
    df["plate_z_norm"] = (df["plate_z"] - df["sz_bot"]) / (df["sz_top"] - df["sz_bot"])
    return df

def normalize_x_for_batter_view(df):
    df = df.copy()
    left_mask = df["stand"] == "L"
    if "plate_x" in df.columns:
        df.loc[left_mask, "plate_x"] = -df.loc[left_mask, "plate_x"]
    return df

def preprocess_pitch_df(df):
    df = df.copy()
    df = add_plate_z_norm(df)
    df = normalize_x_for_batter_view(df)
    return df

def build_course_order():
    strike_order = []

    for r in range(1, N_Z + 1):
        for c in range(N_X, 0, -1):
            strike_order.append(f"strike_r{r}_c{c}")

    ball_order = [
        "ball_upper_right",
        "ball_upper_left",
        "ball_lower_right",
        "ball_lower_left",
    ]

    return strike_order + ball_order

course_order = build_course_order()
course_sort_map = {course: i + 1 for i, course in enumerate(course_order)}

def build_course_master():
    rows = []

    x_width = (ZONE_X_MAX - ZONE_X_MIN) / N_X
    z_width = (ZONE_Z_MAX - ZONE_Z_MIN) / N_Z

    for r in range(1, N_Z + 1):
        for c in range(1, N_X + 1):
            x_left = ZONE_X_MIN + (c - 1) * x_width
            x_right = ZONE_X_MIN + c * x_width
            z_top = ZONE_Z_MAX - (r - 1) * z_width
            z_bottom = ZONE_Z_MAX - r * z_width

            course = f"strike_r{r}_c{c}"

            rows.append({
                "course": course,
                "course_group": "strike_zone",
                "course_row": r,
                "course_col": c,
                "course_sort_order": course_sort_map[course],
                "course_center_x": (x_left + x_right) / 2,
                "course_center_z": (z_bottom + z_top) / 2,
                "course_x_min": x_left,
                "course_x_max": x_right,
                "course_z_min": z_bottom,
                "course_z_max": z_top,
            })

    ball_defs = [
        {
            "course": "ball_upper_right",
            "course_group": "ball_zone",
            "course_row": 0,
            "course_col": 6,
            "x_min": X_CENTER,
            "x_max": BALL_X_MAX,
            "z_min": Z_CENTER,
            "z_max": BALL_Z_MAX,
        },
        {
            "course": "ball_upper_left",
            "course_group": "ball_zone",
            "course_row": 0,
            "course_col": 1,
            "x_min": BALL_X_MIN,
            "x_max": X_CENTER,
            "z_min": Z_CENTER,
            "z_max": BALL_Z_MAX,
        },
        {
            "course": "ball_lower_right",
            "course_group": "ball_zone",
            "course_row": 7,
            "course_col": 6,
            "x_min": X_CENTER,
            "x_max": BALL_X_MAX,
            "z_min": BALL_Z_MIN,
            "z_max": Z_CENTER,
        },
        {
            "course": "ball_lower_left",
            "course_group": "ball_zone",
            "course_row": 7,
            "course_col": 1,
            "x_min": BALL_X_MIN,
            "x_max": X_CENTER,
            "z_min": BALL_Z_MIN,
            "z_max": Z_CENTER,
        },
    ]

    for d in ball_defs:
        rows.append({
            "course": d["course"],
            "course_group": d["course_group"],
            "course_row": d["course_row"],
            "course_col": d["course_col"],
            "course_sort_order": course_sort_map[d["course"]],
            "course_center_x": (d["x_min"] + d["x_max"]) / 2,
            "course_center_z": (d["z_min"] + d["z_max"]) / 2,
            "course_x_min": d["x_min"],
            "course_x_max": d["x_max"],
            "course_z_min": d["z_min"],
            "course_z_max": d["z_max"],
        })

    course_master = pd.DataFrame(rows)
    course_master["course"] = pd.Categorical(
        course_master["course"],
        categories=course_order,
        ordered=True
    )
    course_master = course_master.sort_values("course").reset_index(drop=True)
    course_master["course"] = course_master["course"].astype(str)

    return course_master

course_master = build_course_master()

def assign_course(row):
    x = row["plate_x"]
    z = row["plate_z_norm"]

    in_x = ZONE_X_MIN <= x <= ZONE_X_MAX
    in_z = ZONE_Z_MIN <= z <= ZONE_Z_MAX

    if in_x and in_z:
        x_bin = int(np.floor((x - ZONE_X_MIN) / ((ZONE_X_MAX - ZONE_X_MIN) / N_X)))
        z_bin = int(np.floor((z - ZONE_Z_MIN) / ((ZONE_Z_MAX - ZONE_Z_MIN) / N_Z)))

        x_bin = min(max(x_bin, 0), N_X - 1)
        z_bin = min(max(z_bin, 0), N_Z - 1)

        row_no = N_Z - z_bin

        col_no = x_bin + 1

        return f"strike_r{row_no}_c{col_no}"

    if x >= X_CENTER and z >= Z_CENTER:
        return "ball_upper_right"

    if x < X_CENTER and z >= Z_CENTER:
        return "ball_upper_left"

    if x >= X_CENTER and z < Z_CENTER:
        return "ball_lower_right"

    return "ball_lower_left"

In [6]:
# Load data
all_parts = []
required_cols = [
    TARGET_COLUMN, SPEED_COLUMN,
    "description", "plate_x",
    "plate_z", "sz_bot",
    "sz_top", "stand"]

for year in YEARS:
    year_dir = DATA_ROOT / str(year)
    csv_files = sorted(year_dir.glob("*.csv"))

    print(f"{year}: {len(csv_files)} files")
    for csv_path in tqdm(csv_files, desc=str(year)):
        df = pd.read_csv(csv_path)
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print(f"skip {csv_path.name}: missing columns = {missing}")
            continue

        df = preprocess_pitch_df(df)

        tmp = df[
                [ TARGET_COLUMN, SPEED_COLUMN,
                  "description", "plate_x",
                  "plate_z_norm", "stand",]].copy()

        tmp["year"] = year
        tmp["source_file"] = csv_path.name

        all_parts.append(tmp)

all_df = pd.concat(all_parts, ignore_index=True)


for col in [TARGET_COLUMN, SPEED_COLUMN, "plate_x", "plate_z_norm"]:
    all_df[col] = pd.to_numeric(all_df[col], errors="coerce")

analysis_df = all_df[
    (all_df["description"] == "hit_into_play") &
    (all_df[TARGET_COLUMN] > 0) &
    all_df[TARGET_COLUMN].notna() &
    all_df[SPEED_COLUMN].notna() &
    all_df["plate_x"].notna() &
    all_df["plate_z_norm"].notna()
].copy()

# Divide into three groups based on frequency information
q1 = analysis_df[SPEED_COLUMN].quantile(1 / 3)
q2 = analysis_df[SPEED_COLUMN].quantile(2 / 3)
def assign_speed_group(v):
    if v <= q1:
        return "low_speed"
    elif v <= q2:
        return "middle_speed"
    else:
        return "high_speed"

analysis_df["speed_group"] = analysis_df[SPEED_COLUMN].apply(assign_speed_group)
analysis_df["course"] = analysis_df.apply(assign_course, axis=1)

# Summarize
summary = (
    analysis_df
    .groupby(["speed_group", "course"], as_index=False)
    .agg(
        mean_estimated_slg_using_speedangle=(TARGET_COLUMN, "mean"),
        sample_count=(TARGET_COLUMN, "size"),
        mean_effective_speed=(SPEED_COLUMN, "mean"),
        min_effective_speed=(SPEED_COLUMN, "min"),
        max_effective_speed=(SPEED_COLUMN, "max"),
    )
)

speed_order = ["high_speed", "middle_speed", "low_speed"]
full_index = pd.MultiIndex.from_product(
    [speed_order, course_order],
    names=["speed_group", "course"]
).to_frame(index=False)

summary_full = full_index.merge(
    summary,
    on=["speed_group", "course"],
    how="left")

summary_full = summary_full.merge(
    course_master,
    on="course",
    how="left")

summary_full["sample_count"] = summary_full["sample_count"].fillna(0).astype(int)

summary_full["speed_group"] = pd.Categorical(
    summary_full["speed_group"],
    categories=speed_order,
    ordered=True)

summary_full["course"] = pd.Categorical(
    summary_full["course"],
    categories=course_order,
    ordered=True)

summary_full = summary_full.sort_values(
    ["speed_group", "course"]).reset_index(drop=True)

summary_full["course"] = summary_full["course"].astype(str)
summary_full["speed_group"] = summary_full["speed_group"].astype(str)

summary_full = summary_full[
    [
        "speed_group",
        "course_sort_order",
        "course",
        "course_group",
        "course_row",
        "course_col",
        "course_center_x",
        "course_center_z",
        "course_x_min",
        "course_x_max",
        "course_z_min",
        "course_z_max",
        "mean_estimated_slg_using_speedangle",
        "sample_count",
        "mean_effective_speed",
        "min_effective_speed",
        "max_effective_speed",
    ]
]

speed_thresholds = pd.DataFrame({
    "speed_group_rule": [
        "low_speed: effective_speed <= q33",
        "middle_speed: q33 < effective_speed <= q67",
        "high_speed: q67 < effective_speed"],
    "q33_effective_speed": [q1, q1, q1],
    "q67_effective_speed": [q2, q2, q2]})


output_summary_path = OUTPUT_DIR / "course_estimated_slg_by_speed_2018_2024.csv"
output_course_master_path = OUTPUT_DIR / "course_master_40_areas.csv"
output_threshold_path = OUTPUT_DIR / "speed_group_thresholds_2018_2024.csv"

summary_full.to_csv(output_summary_path, index=False)
course_master.to_csv(output_course_master_path, index=False)
speed_thresholds.to_csv(output_threshold_path, index=False)

print("\nSaved the data:")
print(f"low_speed    : effective_speed <= {q1:.3f}")
print(f"middle_speed : {q1:.3f} < effective_speed <= {q2:.3f}")
print(f"high_speed   : {q2:.3f} < effective_speed")


2018: 57 files


2018:   0%|          | 0/57 [00:00<?, ?it/s]

2019: 61 files


2019:   0%|          | 0/61 [00:00<?, ?it/s]

2022: 45 files


...

2023: 44 files


2023:   0%|          | 0/44 [00:00<?, ?it/s]

2024: 58 files


2024:   0%|          | 0/58 [00:00<?, ?it/s]


Saved the data:
low_speed    : effective_speed <= 86.700
middle_speed : 86.700 < effective_speed <= 92.500
high_speed   : 92.500 < effective_speed


In [7]:
from google.colab import runtime
runtime.unassign()